# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 18: Rate Limiting, Throttling y Prevención de Abuso
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Contador manual de peticiones por IP en memoria*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Rate Limiter deslizante con encabezados Retry-After* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de ráfaga masiva y bloqueo temporal (HTTP 429)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion18.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [1]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
⏳ Instalando PHP CLI...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno Web listo:
Python 3.13.18
PHP 8.3.6 (cli) (built: Nov 18 2026 23:42:07) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Contador manual de peticiones por IP en memoria

In [2]:
%%writefile app_artesano.py
import time
from fastapi import FastAPI, HTTPException, Request, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

MAX_PETICIONES = 5
VENTANA = 10  # segundos
visitas_por_ip = {}

class PayloadRequest(BaseModel):
    dato: str

@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest, request: Request):
    ip = request.client.host if request.client else "unknown"
    ahora = time.time()

    # limpiamos registros fuera de la ventana
    historial = [t for t in visitas_por_ip.get(ip, []) if (ahora - t) < VENTANA]

    if len(historial) >= MAX_PETICIONES:
        visitas_por_ip[ip] = historial
        raise HTTPException(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            detail="Demasiadas peticiones"
        )

    historial.append(ahora)
    visitas_por_ip[ip] = historial

    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")

    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}

Writing app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [4]:
%%writefile app_ia.py
import math
import threading
import time
from collections import deque
from fastapi import FastAPI, Request, status
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(title="API Cientifico - Sliding Window")

MAX_PETICIONES = 5
VENTANA = 10  # segundos

_lock = threading.Lock()
_ventanas_ip: dict[str, deque[float]] = {}


class PayloadRequest(BaseModel):
    dato: str


def _evaluar_limite(ip: str, ahora: float) -> tuple[bool, int, int, int]:
    """
    Evalúa ventana deslizante por IP.
    Retorna: (permitido, restantes, retry_after_seg, reset_epoch)
    """
    with _lock:
        cola = _ventanas_ip.setdefault(ip, deque())

        # Elimina marcas de tiempo fuera de la ventana deslizante
        limite_inferior = ahora - VENTANA
        while cola and cola[0] <= limite_inferior:
            cola.popleft()

        # Limpia la clave si no hay registros activos
        if not cola and ip in _ventanas_ip:
            pass

        if len(cola) >= MAX_PETICIONES:
            # Cuándo se libera el registro más antiguo dentro de la ventana
            espera = (cola[0] + VENTANA) - ahora
            retry_after = max(1, math.ceil(espera))
            reset_epoch = math.ceil(cola[0] + VENTANA)
            return False, 0, retry_after, reset_epoch

        cola.append(ahora)
        restantes = max(0, MAX_PETICIONES - len(cola))
        reset_epoch = math.ceil(cola[0] + VENTANA)
        return True, restantes, 0, reset_epoch


@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest, request: Request):
    ip = request.client.host if request.client else "unknown"
    ahora = time.time()

    permitido, restantes, retry_after, reset_epoch = _evaluar_limite(ip, ahora)

    headers_base = {
        "X-RateLimit-Limit": str(MAX_PETICIONES),
        "X-RateLimit-Remaining": str(restantes),
        "X-RateLimit-Reset": str(reset_epoch),
    }

    if not permitido:
        headers_429 = {
            **headers_base,
            "Retry-After": str(retry_after),
        }
        return JSONResponse(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            content={
                "detail": "Demasiadas peticiones",
                "retry_after": retry_after,
            },
            headers=headers_429,
        )

    if not payload.dato.strip():
        return JSONResponse(
            status_code=status.HTTP_400_BAD_REQUEST,
            content={"detail": "Dato vacio"},
            headers=headers_base,
        )

    return JSONResponse(
        status_code=status.HTTP_200_OK,
        content={
            "status": "success",
            "mensaje": f"Procesado: {payload.dato}",
            "remaining": restantes,
        },
        headers=headers_base,
    )


Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de ráfaga masiva y bloqueo temporal (HTTP 429)

In [7]:
import threading
import time
import httpx
import uvicorn

from app_artesano import app as app_artesano
from app_ia import app as app_ia


def arrancar_api(app_obj, puerto):
    cfg = uvicorn.Config(app_obj, host="127.0.0.1", port=puerto, log_level="error")
    srv = uvicorn.Server(cfg)
    threading.Thread(target=srv.run, daemon=True).start()
    time.sleep(1)


def probar_rafaga(url, total=8):
    with httpx.Client(base_url=url, timeout=5.0) as cliente:
        for i in range(1, total + 1):
            r = cliente.post("/api/recurso", json={"dato": f"test-{i}"})
            retry = r.headers.get("retry-after")
            print(f"Petición {i}: {r.status_code}" + (f" (Retry-After: {retry}s)" if retry else ""))

        # comprobar bloqueo seguido
        r_bloqueo = cliente.post("/api/recurso", json={"dato": "extra"})
        print("Bloqueo inmediato:", r_bloqueo.status_code)

        # esperar ventana de 10s y reintentar
        time.sleep(11)
        r_ok = cliente.post("/api/recurso", json={"dato": "despues-espera"})
        print("Después de esperar:", r_ok.status_code, r_ok.json())


arrancar_api(app_artesano, 8011)
arrancar_api(app_ia, 8012)

print("--- Artesano ---")
probar_rafaga("http://127.0.0.1:8011")

print("--- IA ---")
probar_rafaga("http://127.0.0.1:8012")

--- Artesano ---
Petición 1: 200
Petición 2: 200
Petición 3: 200
Petición 4: 200
Petición 5: 200
Petición 6: 429
Petición 7: 429
Petición 8: 429
Bloqueo inmediato: 429
Después de esperar: 200 {'status': 'success', 'mensaje': 'Procesado: despues-espera'}
--- IA ---
Petición 1: 200
Petición 2: 200
Petición 3: 200
Petición 4: 200
Petición 5: 200
Petición 6: 429 (Retry-After: 10s)
Petición 7: 429 (Retry-After: 10s)
Petición 8: 429 (Retry-After: 10s)
Bloqueo inmediato: 429
Después de esperar: 200 {'status': 'success', 'mensaje': 'Procesado: despues-espera', 'remaining': 4}
